In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
schema = StructType([
  StructField("unique_key", StringType(),False),
  StructField("created_date",TimestampType()),
  StructField("closed_date",TimestampType()),
  StructField("name",StringType()),
  StructField("agency",StringType()),
  StructField("agency_name",StringType()),
  StructField("complaint_type",StringType()),
  StructField("descriptor",StringType()),
  StructField("descriptor_2",StringType()),
  StructField("location_type",StringType()),
  StructField("incident_zip",StringType()),
  StructField("incident_address",StringType()),
  StructField("street_name",StringType()),
  StructField("cross_street_1",StringType()),
  StructField("cross_street_2",StringType()),
  StructField("intersection_street_1",StringType()),
  StructField("intersection_street_2",StringType()),
  StructField("address_type",StringType()),
  StructField("city",StringType()),
  StructField("landmark",StringType()),
  StructField("facility_type",StringType()),
  StructField("status",StringType()),
  StructField("due_date",TimestampType()),
  StructField("resolution_description",StringType()),
  StructField("resolution_action_updated_date",TimestampType()),
  StructField("community_board",StringType()),
  StructField("council_district",StringType()),
  StructField("fbicr_code",StringType()),
  StructField("police_precinct",StringType()),
  StructField("latitude",StringType()),
  StructField("bbl",StringType()),
  StructField("borough",StringType()),
  StructField("location",StringType()),
  StructField("x_coordinate",StringType()),
  StructField("y_coordinate",StringType()),
  StructField("longitude",StringType()),
  StructField("open_data_channel_type",StringType()),
  StructField("park_facility_name",StringType()),
  StructField("park_borough",StringType()),
  StructField("vehicle_type",StringType()),
  StructField("taxi_company_borough",StringType()),
  StructField("taxi_pick_up_location",StringType()),
  StructField("bridge_highway_name",StringType()),
  StructField("bridge_highway_direction",StringType()),
  StructField("road_ramp",StringType()),
  StructField("bridge_highway_segment",StringType())
])
df = spark.createDataFrame([], schema)

In [0]:
display(dbutils.fs.ls("/Volumes/workspace/default/nyc311_raw/"))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/nyc311_raw/page_1.json,page_1.json,80653866,1790972740000
dbfs:/Volumes/workspace/default/nyc311_raw/page_10.json,page_10.json,83099416,1790975560000
dbfs:/Volumes/workspace/default/nyc311_raw/page_11.json,page_11.json,81698800,1790975723000
dbfs:/Volumes/workspace/default/nyc311_raw/page_12.json,page_12.json,82769293,1790975760000
dbfs:/Volumes/workspace/default/nyc311_raw/page_13.json,page_13.json,82938299,1790975931000
dbfs:/Volumes/workspace/default/nyc311_raw/page_14.json,page_14.json,81870274,1790976054000
dbfs:/Volumes/workspace/default/nyc311_raw/page_15.json,page_15.json,84925780,1790977327000
dbfs:/Volumes/workspace/default/nyc311_raw/page_16.json,page_16.json,81893457,1790977351000
dbfs:/Volumes/workspace/default/nyc311_raw/page_17.json,page_17.json,81094213,1790976985000
dbfs:/Volumes/workspace/default/nyc311_raw/page_18.json,page_18.json,82549772,1790977018000


In [0]:
raw_df = spark.read.json(
    "/Volumes/workspace/default/nyc311_raw/"
)

print(raw_df.count())
raw_df.show(5)

1750000
+---------------------------+---------------------------+---------------------------+---------------------------+------------+------+--------------------+----------+---------+------------------------+-------------------+----------------------+-------------------+--------------------+---------------+--------------------+----------------+--------------------+---------------+----------------+--------------------+----------------+--------+-------------+--------------------+------------+---------------------+---------------------+---------------+------------------+--------------------+---------------+------------------+----------------------+------------+------------------+---------------+------------------------------+----------------------+---------+------+-----------------+--------------------+---------------------+----------+------------+------------------------+------------------------+
|:@computed_region_92fq_4b7q|:@computed_region_f5dn_yrer|:@computed_region_sbqj_enih|:@compu

In [0]:
from pyspark.sql.functions import current_timestamp, col

bronze_df = raw_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
).withColumn(
    "source_file",
    col("_metadata.file_path")
)

print(bronze_df)

DataFrame[:@computed_region_92fq_4b7q: string, :@computed_region_f5dn_yrer: string, :@computed_region_sbqj_enih: string, :@computed_region_yeji_bk3q: string, address_type: string, agency: string, agency_name: string, bbl: string, borough: string, bridge_highway_direction: string, bridge_highway_name: string, bridge_highway_segment: string, city: string, closed_date: string, community_board: string, complaint_type: string, council_district: string, created_date: string, cross_street_1: string, cross_street_2: string, descriptor: string, descriptor_2: string, due_date: string, facility_type: string, incident_address: string, incident_zip: string, intersection_street_1: string, intersection_street_2: string, landmark: string, latitude: string, location: struct<coordinates:array<double>,type:string>, location_type: string, longitude: string, open_data_channel_type: string, park_borough: string, park_facility_name: string, police_precinct: string, resolution_action_updated_date: string, res

In [0]:
raw_df = spark.read.schema(schema).json(
    "/Volumes/workspace/default/nyc311_raw/"
)

In [0]:
from pyspark.sql.functions import current_timestamp, col

bronze_df = raw_df.withColumn(
    "ingestion_timestamp",
    current_timestamp()
).withColumn(
    "source_file",
    col("_metadata.file_path")
)

bronze_df.show(5)

+----------+-------------------+-------------------+----+------+--------------------+--------------------+--------------------+----------------+---------------+------------+--------------------+-----------------+---------------+----------------+---------------------+---------------------+------------+-------------------+---------------+-------------+------+--------+----------------------+------------------------------+---------------+----------------+----------+---------------+------------------+----------+---------+--------------------+------------+------------+------------------+----------------------+------------------+------------+------------+--------------------+---------------------+-------------------+------------------------+---------+----------------------+--------------------+--------------------+
|unique_key|       created_date|        closed_date|name|agency|         agency_name|      complaint_type|          descriptor|    descriptor_2|  location_type|incident_zip|    inc

In [0]:
print("Rows:", bronze_df.count())
print("Columns:", len(bronze_df.columns))

bronze_df.groupBy("unique_key").count().filter("count > 1").show()

Rows: 1750000
Columns: 48
+----------+-----+
|unique_key|count|
+----------+-----+
|  69507060|    2|
|  69499061|    2|
|  69503329|    2|
|  69504328|    2|
|  69494655|    2|
|  69494842|    2|
|  69498770|    2|
|  69502958|    2|
|  69495376|    2|
|  69498692|    2|
|  69500200|    2|
|  69496317|    2|
|  69503243|    2|
|  69497534|    2|
|  69503578|    2|
|  69503922|    2|
|  69506775|    2|
|  69496828|    2|
|  69495226|    2|
|  69494155|    2|
+----------+-----+
only showing top 20 rows


originally there were 46 columns in the dataset but after adding the ingestion_timestamp and source_file we have 48 columns 

In [0]:
bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "workspace.default.bronze_nyc311"
)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType, DoubleType

silver_schema = StructType([
  StructField("unique_key", StringType(), False),
  StructField("created_date", TimestampType()),
  StructField("closed_date", TimestampType()),
  StructField("name", StringType()),
  StructField("agency", StringType()),
  StructField("agency_name", StringType()),
  StructField("complaint_type", StringType()),
  StructField("descriptor", StringType()),
  StructField("descriptor_2", StringType()),
  StructField("location_type", StringType()),
  StructField("incident_zip", StringType()),
  StructField("incident_address", StringType()),
  StructField("street_name", StringType()),
  StructField("cross_street_1", StringType()),
  StructField("cross_street_2", StringType()),
  StructField("intersection_street_1", StringType()),
  StructField("intersection_street_2", StringType()),
  StructField("address_type", StringType()),
  StructField("city", StringType()),
  StructField("landmark", StringType()),
  StructField("facility_type", StringType()),
  StructField("status", StringType()),
  StructField("due_date", TimestampType()),
  StructField("resolution_description", StringType()),
  StructField("resolution_action_updated_date", TimestampType()),
  StructField("community_board", StringType()),
  StructField("council_district", IntegerType()),
  StructField("fbicr_code", StringType()),
  StructField("police_precinct", StringType()),
  StructField("latitude", DoubleType()),
  StructField("bbl", StringType()),
  StructField("borough", StringType()),
  StructField("location", StringType()),
  StructField("x_coordinate", DoubleType()),
  StructField("y_coordinate", DoubleType()),
  StructField("longitude", DoubleType()),
  StructField("open_data_channel_type", StringType()),
  StructField("park_facility_name", StringType()),
  StructField("park_borough", StringType()),
  StructField("vehicle_type", StringType()),
  StructField("taxi_company_borough", StringType()),
  StructField("taxi_pick_up_location", StringType()),
  StructField("bridge_highway_name", StringType()),
  StructField("bridge_highway_direction", StringType()),
  StructField("road_ramp", StringType()),
  StructField("bridge_highway_segment", StringType()),
  StructField("load_timestamp", TimestampType()),
  StructField("source_file", StringType())
])

In [0]:
from pyspark.sql.functions import col, trim
silver_df = bronze_df.filter(
    col("unique_key").isNotNull()
)

and the above filter has removed all the null values in the unique_key column

In [0]:
silver_df = silver_df.dropDuplicates(["unique_key"])


the above unique key and count shows that what unique key is repeating and how many times and in the silver layer, i have removed those rows as they are duplicatting rows.

In [0]:

silver_df = silver_df.withColumn(
    "council_district",
    col("council_district").cast("integer")
).withColumn(
    "latitude",
    col("latitude").cast("double")
).withColumn(
    "longitude",
    col("longitude").cast("double")
).withColumn(
    "x_coordinate",
    col("x_coordinate").cast("double")
).withColumn(
    "y_coordinate",
    col("y_coordinate").cast("double")
)

silver_df = silver_df.withColumn("agency",trim(col("agency"))).withColumn(
    "agency_name",
    trim(col("agency_name"))
).withColumn(
    "complaint_type",
    trim(col("complaint_type"))
).withColumn(
    "status",
    trim(col("status"))
).withColumn(
    "borough",
    trim(col("borough"))
)